### PyTorch Lightning image classification model ###
This notebook uses the ToothModel lightning class in src/models

In [1]:
import sys
import os
import copy
import glob
import json
import pandas as pd
import numpy as np
from pathlib import Path
import albumentations as alb
from matplotlib import pyplot as plt
from dotenv import load_dotenv

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights

# Lightning module
from lightning.pytorch import LightningModule, Trainer, seed_everything
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

# Appearance of the Notebook
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Import this module with autoreload
%load_ext autoreload
%autoreload 2

import computervision as cv
from computervision.fileutils import FileOP
from computervision.inference import get_gpu_info
from computervision.imageproc import ImageData, is_image
from computervision.transformations import AugmentationTransform
from computervision.datasets import DatasetFromDF
from computervision.models.lightningmodel import ToothModel

# Print version info
print(f'Package version: {cv.__version__}')
print(f'Authors:         {cv.__authors__}')
print(f'Python version:  {sys.version}')

Package version: v0.0.2
Authors:         The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.11 (main, Sep 14 2025, 07:49:30) [GCC 13.3.0]


In [2]:
# Load environmental variables
load_dotenv()
data_root = os.environ.get('DATA_DIR')
if data_root is None:
    print('Edit the "env"file and copy it to the root of the repository.') 
    raise FileNotFoundError('No .env file found. The data directory was not set.')

# Directory to store the data
dataset_name = 'computervision_data'
data_dir = os.path.join(data_root, dataset_name)
example_dir = os.path.join(data_dir, 'examples')

# URL with the data set for manual download
url = os.environ.get('CL_URL')
image_dir_name = os.path.basename(url).split('.')[0]
image_dir = os.path.join(data_dir, image_dir_name, 'cropped')

# Verify the image data on disk
file_list = glob.glob(os.path.join(image_dir, '*.jpg'))
print(len(file_list))

# Set up a model directory for the trained model
model_dir = os.path.join(data_dir, 'model')
Path(model_dir).mkdir(exist_ok=True, parents=True)

7697


### Load annotations ###

In [3]:
annotations_file_name = 'annotations_cropped_dset.parquet'
annotations_file = os.path.join(image_dir, annotations_file_name)
df = pd.read_parquet(annotations_file)

file_col = 'file_name'
bbox_col= 'bbox'
label_col = 'label'
dset_col = 'dset'

labels = list(df[label_col].unique())
label2id = dict(zip(labels, range(len(labels))))
id2label = {category_id: label for label, category_id in label2id.items()}
display(label2id)

# Now we can add a category id to the data frame
df = df.assign(category=df[label_col].apply(lambda label: label2id.get(label)))

# Check the images
file_list = [os.path.join(image_dir, file_name) for file_name in df[file_col].unique()]
checked = [is_image(file) for file in file_list]
assert len(file_list) == sum(checked), f'WARNING: Could not open all {len(file_list)} images at: {image_dir}'
print(f'Image directory:        {image_dir}')
print(f'Total number of images: {len(file_list)}')
print(f'Annotations:            {df.shape[0]}')
display(df.head())

{'tooth': 0,
 'amalgam': 1,
 'caries': 2,
 'composite': 3,
 'Calculus': 4,
 'root filling': 5}

Image directory:        /home/andreas/data/computervision_data/dataset_dental_roboflow/cropped
Total number of images: 7550
Annotations:            7550


,multi_file,file_name,width,height,area,label,bbox,pos,pos_bbox,dset,category
0,a0ab0bec5c.jpg,tooth_02_a0ab0bec5c.jpg,152,236,35872,tooth,None,[2],None,train,0
1,5c6869fdf0.jpg,amalgam_03_5c6869fdf0.jpg,427,265,113155,amalgam,"[182, 29, 119, 112]","[19, 20]","[[143, 20, 284, 245], [20, 33, 382, 232]]",train,1
2,96a9546c1e.jpg,caries_08_96a9546c1e.jpg,525,253,132825,caries,"[97, 97, 46, 32]","[19, 20]","[[129, 21, 396, 232], [20, 20, 256, 233]]",train,2
3,1e36413c9e.jpg,composite_05_1e36413c9e.jpg,535,241,128935,composite,"[67, 32, 169, 87]","[29, 30]","[[213, 44, 322, 197], [20, 20, 341, 221]]",train,3
4,872cd0d6f2.jpg,tooth_15_872cd0d6f2.jpg,213,181,38553,tooth,None,[15],None,train,0


### Set up the image augmentations ###

In [4]:
# Initial scaling and padding for the bigger dimension
max_image_size = 640

# Model input size
im_width, im_height = 224, 224
train_transforms = AugmentationTransform(im_width=im_width, im_height=im_height).\
                get_transforms(name='train_transform')

# Resize and then normalize
# with ImageNet mean and standard deviation for ResNet50
image_net_mean = ImageData().image_net_mean
image_net_std = ImageData().image_net_std

# This transform is essential and needs to be applied for both training and validation
resize_and_normalize = [alb.Resize(width=im_width, height=im_height),
                        alb.Normalize(mean=image_net_mean, std=image_net_std)]
train_transforms.extend(resize_and_normalize)
train_transform = alb.Compose(train_transforms)

# However, for validation and testing, we don't want the augmentations,
# so we just resize and normalize the data
test_transform = alb.Compose(resize_and_normalize)

### Datasets ###

In [5]:
train_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'train'],
                              image_dir=image_dir,
                              file_name_col=file_col,
                              label_id_col='category',
                              max_image_size=max_image_size,
                              transform=train_transform,
                              validate=True)


val_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'val'],
                            image_dir=image_dir,
                            file_name_col=file_col,
                            label_id_col='category',
                            max_image_size=max_image_size,
                            transform=test_transform,
                            validate=True)

test_dataset = DatasetFromDF(data=df.loc[df[dset_col] == 'test'],
                             image_dir=image_dir,
                             file_name_col=file_col,
                             label_id_col='category',
                             max_image_size=max_image_size,
                             transform=test_transform,
                             validate=True)

### Model parameters ###

In [9]:
device_number = 0
device, device_str = get_gpu_info(device_number=device_number)

model_name = 'lightningmodel'
model_version = 0
model_version_str = str(model_version).zfill(2)

model_name_dir = os.path.join(model_dir, model_name)
checkpoint_dir = os.path.join(model_name_dir, f'{model_name}_{model_version_str}')
Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

model_info = {'model_version': model_version_str,
              'device_number': device_number,
              'project_version': cv.__version__,
              'model_name': model_name,
              'image_dir': image_dir,
              'model_dir': checkpoint_dir,
              'im_width': im_width,
              'im_height': im_height,
              'max_image_size': max_image_size}

training_args = {'max_epochs': 5,
                 'num_classes': 6,
                 'num_workers': 4,
                 'batch_size': 60,
                 'initial_lr': 1.0e-3,
                 'check_val_every_n_epoch': 1,
                 'checkpoint_very_n_epoch': 2,
                 'save_top_k': 3}

# Save the model parameters
parameters = {'model_info': model_info,
              'id2label': id2label,
              'training_args': training_args}

json_file = os.path.join(checkpoint_dir, f'{model_name}.json')
with open(json_file, 'w') as f:
    json.dump(parameters, f, indent=4)

Current device:    cuda:0


### Create the model instance ###

In [10]:
model = ToothModel(train_dataset=train_dataset,
                   val_dataset=val_dataset,
                   test_dataset=test_dataset,
                   batch_size=training_args.get('batch_size'),
                   num_classes=training_args.get('num_classes'),
                   num_workers=training_args.get('num_workers'),
                   lr=training_args.get('initial_lr'))

### Set up TensorBoard logger and ModelCheckpoint callbacks ###

In [11]:
# Directory to save checkpoints and logs
chk_callback = ModelCheckpoint(dirpath=checkpoint_dir,
                               filename='model-{epoch}',
                               monitor='val_loss',
                               mode='min',
                               save_last=True,
                               every_n_epochs=training_args.get('checkpoint_every_n_epoch'),
                               save_on_train_epoch_end=True,
                               save_top_k=training_args.get('save_top_k'))

# Setup logger
logger = TensorBoardLogger(save_dir=checkpoint_dir,
                           name='log')

lr_monitor = LearningRateMonitor(logging_interval='epoch',
                                 log_momentum=True)

### Run the training loop ###

In [12]:
seed_everything(42)
tr = Trainer(max_epochs=training_args.get('max_epochs'),
             default_root_dir=checkpoint_dir,
             callbacks=[chk_callback, lr_monitor],
             logger=logger,
             check_val_every_n_epoch=training_args.get('check_val_every_n_epoch'))

tr.fit(model)

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/andreas/gitrepos/computervision/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/andreas/data/computervision_data/model/lightningmodel/lightningmodel_00 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ResNet           │ 24.6 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ metrics   │ ModuleDict       │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 24.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.6 M                                                                                               
Total estimated model params size (MB): 98                                                                         
Modules in train mode: 161                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/andreas/gitrepos/computervision/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21:
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/home/andreas/gitrepos/computervision/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: 
UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in 
true positive score
  warnings.warn(*args, **kwargs)

`Trainer.fit` stopped: `max_epochs=5` reached.
